# Day 1 · Section 5: Token Embeddings

Standalone student notebook. Run cells in order, change the examples, and use the checks to explain what happened. It contains no workshop slides. CPU exercises work without downloads; optional Qwen3 cells require network access and, where indicated, a suitable GPU.


## Goals · 5.1–5.4

Map token IDs to rows of a learned embedding table, track `[batch, sequence, hidden]` shapes, inspect Qwen3's real table when a GPU is available, and distinguish input token embeddings from document-level retrieval embeddings.


In [ ]:
import sys, subprocess, numpy as np
try:
    import torch
    DEVICE=torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    print('PyTorch',torch.__version__,'device',DEVICE)
except ImportError:
    torch=None;DEVICE=None
    print('PyTorch unavailable locally; NumPy path remains runnable. Colab normally includes PyTorch.')
rng=np.random.default_rng(7)


In [ ]:
table=np.array([[1.,0.,0.],[0.,1.,0.],[0.,0.,1.],[.5,.5,.5]],dtype=np.float32)
ids=np.array([[3,1,2,3],[0,2,1,0]],dtype=np.int64)
vectors=table[ids]
print('IDs',ids.shape,'table',table.shape,'lookup',vectors.shape)
assert vectors.shape==(2,4,3)
assert np.array_equal(vectors[0,0],vectors[0,3])
print('same ID → same input row:',vectors[0,0])


### Batch, sequence and hidden axes

Reshape only when you know what each axis represents. Summing over sequence positions produces one simple document-like vector per item, while preserving sequence keeps one vector per token.


In [ ]:
pooled=vectors.mean(axis=1)
print('sequence vectors:',vectors.shape,'mean-pooled vectors:',pooled.shape)
assert pooled.shape==(2,3)
if torch is not None:
    table_t=torch.tensor(table,device=DEVICE)
    ids_t=torch.tensor(ids,dtype=torch.long,device=DEVICE)
    lookup_t=torch.nn.functional.embedding(ids_t,table_t)
    print('PyTorch lookup',tuple(lookup_t.shape),'pooled',tuple(lookup_t.mean(dim=1).shape))
    assert np.allclose(lookup_t.cpu().numpy(),vectors)


### Context changes a token's later representation

An input lookup is context-independent. The tiny mixing matrix below is a stand-in for a contextual layer: each position reads other positions. It is not a learned Transformer.


In [ ]:
sentence_a=table[np.array([3,1,2])]
sentence_b=table[np.array([1,3,2])]
mix=np.array([[.7,.3,0.],[.2,.6,.2],[0.,.3,.7]])
state_a=mix@sentence_a;state_b=mix@sentence_b
print('same ID 3 input row:',sentence_a[0],sentence_b[1])
print('contextual outputs differ:',state_a[0],state_b[1])
assert not np.allclose(state_a[0],state_b[1])


### 5.2 · Inspect Qwen3 embeddings (optional GPU)

This downloads the 4B model. The default gate requires at least 10 GiB free GPU memory. Its input embedding table has vocabulary rows × hidden features; the tokenizer base vocabulary can have fewer entries than the table.


In [ ]:
import sys, subprocess
try:
    import torch
    HAS_GPU=torch.cuda.is_available()
except ImportError:
    torch=None;HAS_GPU=False
RUN_QWEN=HAS_GPU  # set False to skip the multi-GB download
if HAS_GPU:
    print('GPU:',torch.cuda.get_device_name(0),'free GiB:',round(torch.cuda.mem_get_info()[0]/2**30,2))
else:print('No CUDA GPU; Qwen3 model cells will be skipped.')


In [ ]:
model=tokenizer=None
if RUN_QWEN and torch.cuda.mem_get_info()[0]/2**30>=10:
    subprocess.check_call([sys.executable,'-m','pip','-q','install','transformers>=4.52.4,<6','accelerate','safetensors'])
    from transformers import AutoTokenizer, AutoModelForCausalLM
    tokenizer=AutoTokenizer.from_pretrained('Qwen/Qwen3-4B')
    dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    model=AutoModelForCausalLM.from_pretrained('Qwen/Qwen3-4B',torch_dtype=dtype,device_map='auto')
    model.eval()
    print('Loaded model on:',model.device)
else:print('Qwen3 skipped; default gate requires a CUDA GPU with at least 10 GiB free.')


In [ ]:
if model is not None:
    text='Find the refund policy.'
    inputs=tokenizer(text,return_tensors='pt').to(model.device)
    with torch.inference_mode():real_vectors=model.get_input_embeddings()(inputs['input_ids'])
    print('IDs:',tuple(inputs['input_ids'].shape))
    print('table:',tuple(model.get_input_embeddings().weight.shape))
    print('sequence vectors:',tuple(real_vectors.shape))
    print('one vector, first five values:',real_vectors[0,0,:5].float().cpu().numpy())
else:print('Real Qwen3 embedding lookup skipped.')


### 5.4 · Retrieval embeddings represent larger passages

This toy mean-pooling shows a change in *unit of representation*, not a good retrieval model. Day 2 uses a separately trained text embedding model for chunks and queries.


In [ ]:
chunks=[np.array([3,1,2]),np.array([0,2,1])]
chunk_vectors=np.stack([table[chunk].mean(axis=0) for chunk in chunks])
print('two chunk-level vectors:',chunk_vectors.shape)
assert chunk_vectors.shape==(2,table.shape[1])
# Exercise: compare a two-token chunk with a longer chunk containing the same tokens plus noise.


## Checks

1. What determines the number of embedding rows and vector width?
2. Why does changing a token's surrounding words leave its input row unchanged but affect later contextual states?
3. What information is lost in the toy mean pool? Why use a retrieval embedding model instead?
